# Constellation Diagrams

This notebook turns digital modulation into points on the I/Q plane. As noise rises, clusters spread, decision boundaries blur, and bit errors become more likely.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## QPSK and 16-QAM Under Noise

Constellation diagrams are just scatter plots of complex symbols. Their geometry makes it easy to see why denser alphabets demand more SNR.

In [ ]:
rng = np.random.default_rng(42)
qpsk = np.array([1 + 1j, -1 + 1j, -1 - 1j, 1 - 1j]) / np.sqrt(2)
qam16 = np.array([x + 1j * y for x in (-3, -1, 1, 3) for y in (-3, -1, 1, 3)]) / np.sqrt(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def update_constellation(noise_sigma=0.1):
    for ax in axes:
        ax.clear()
    qpsk_samples = rng.choice(qpsk, size=600) + noise_sigma * (rng.normal(size=600) + 1j * rng.normal(size=600))
    qam_samples = rng.choice(qam16, size=600) + noise_sigma * (rng.normal(size=600) + 1j * rng.normal(size=600))
    axes[0].scatter(qpsk_samples.real, qpsk_samples.imag, s=12, alpha=0.6)
    axes[1].scatter(qam_samples.real, qam_samples.imag, s=12, alpha=0.6, color="tab:orange")
    axes[0].set_title("QPSK")
    axes[1].set_title("16-QAM")
    for ax in axes:
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        ax.set_xlabel("I")
        ax.set_ylabel("Q")
        ax.axhline(0, color="black", linewidth=0.5)
        ax.axvline(0, color="black", linewidth=0.5)
    axes[1].set_xlim(-1.8, 1.8)
    axes[1].set_ylim(-1.8, 1.8)
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_constellation,
    noise_sigma=float_slider(min_value=0.01, max_value=0.6, step=0.01, value=0.1, description="Noise"),
)
display(controls)


## BER Intuition

When noise spreads symbols across the wrong decision region, a bit error occurs. Dense constellations pack more bits per symbol, but their boundaries are closer together.

In [ ]:
out = widgets.Output()

def estimate_qpsk_ber(noise_sigma=0.1, n=20_000):
    bits = rng.integers(0, 2, size=(n, 2))
    mapping = {(0, 0): -1 - 1j, (0, 1): -1 + 1j, (1, 1): 1 + 1j, (1, 0): 1 - 1j}
    symbols = np.array([mapping[tuple(pair)] for pair in bits]) / np.sqrt(2)
    noisy = symbols + noise_sigma * (rng.normal(size=n) + 1j * rng.normal(size=n))
    decided = np.column_stack((noisy.real > 0, noisy.imag > 0)).astype(int)
    remap = {(0, 0): np.array([0, 0]), (0, 1): np.array([0, 1]), (1, 1): np.array([1, 1]), (1, 0): np.array([1, 0])}
    decoded = np.array([remap[tuple(pair)] for pair in decided])
    ber = np.mean(bits != decoded)
    with out:
        out.clear_output(wait=True)
        print(f"Estimated QPSK BER: {ber:.5f}")

controls = widgets.interactive(
    estimate_qpsk_ber,
    noise_sigma=float_slider(min_value=0.01, max_value=0.8, step=0.01, value=0.1, description="Noise"),
)
display(controls, out)


## Key Takeaway

Constellations make digital tradeoffs visible. More bits per symbol improve efficiency, but they also make every noise burst and phase error more expensive.